In [ ]:

from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

# Find project root
current_dir = Path.cwd().resolve()

ROOT_DIR = next(
    (p for p in [current_dir, *current_dir.parents]
     if (p / "config.py").exists()),
    None
)

if ROOT_DIR is None:
    raise FileNotFoundError("Project root not found!")

# Load training history
history_path = (
    ROOT_DIR / "outputs" / "metrics" / "baseline_history.csv"
)

df = pd.read_csv(history_path)

print(df.tail())

# Find the epoch with highest validation Dice
best_row = df.loc[df["val_dice"].idxmax()]

print("\nBest Epoch:", int(best_row["epoch"]))
print(f"Best Validation Dice: {best_row['val_dice']:.4f}")
print("Total Epochs:", len(df))

In [ ]:

PLOTS_DIR = ROOT_DIR / "outputs" / "plots"
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

plt.figure(figsize=(9, 5))

plt.plot(
    df["epoch"],
    df["train_loss"],
    label="Training Loss",
    linewidth=2
)

plt.plot(
    df["epoch"],
    df["val_loss"],
    label="Validation Loss",
    linewidth=2
)

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Baseline U-Net: Training vs Validation Loss")

plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()

plt.savefig(
    PLOTS_DIR / "baseline_loss.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:

best_epoch = int(best_row["epoch"])
best_dice = best_row["val_dice"]

plt.figure(figsize=(9, 5))

plt.plot(
    df["epoch"],
    df["val_dice"],
    label="Validation Dice",
    linewidth=2
)

plt.scatter(
    best_epoch,
    best_dice,
    color="red",
    s=70,
    zorder=5,
    label=f"Best: {best_dice:.4f}"
)

plt.axvline(
    best_epoch,
    color="red",
    linestyle="--",
    alpha=0.5
)

plt.xlabel("Epoch")
plt.ylabel("Dice Score")
plt.title("Baseline U-Net: Validation Dice")

plt.ylim(0, 1)
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()

plt.savefig(
    PLOTS_DIR / "baseline_dice.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:

import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path

# Find project root
current_dir = Path.cwd().resolve()

ROOT_DIR = next(
    (p for p in [current_dir, *current_dir.parents]
     if (p / "config.py").exists()),
    None
)

if ROOT_DIR is None:
    raise FileNotFoundError("Project root not found!")

# Paths
METRICS_DIR = ROOT_DIR / "outputs" / "metrics"
PLOTS_DIR = ROOT_DIR / "outputs" / "plots"

PLOTS_DIR.mkdir(parents=True, exist_ok=True)

# Load training histories
baseline_df = pd.read_csv(
    METRICS_DIR / "baseline_v2_history.csv"
)

augmented_df = pd.read_csv(
    METRICS_DIR / "augmented_history.csv"
)

# Find best epochs
baseline_best = baseline_df.loc[
    baseline_df["val_dice"].idxmax()
]

augmented_best = augmented_df.loc[
    augmented_df["val_dice"].idxmax()
]

print("Baseline V2")
print("Best Epoch:", int(baseline_best["epoch"]))
print(f"Best Dice: {baseline_best['val_dice']:.4f}")

print("\nAugmented U-Net")
print("Best Epoch:", int(augmented_best["epoch"]))
print(f"Best Dice: {augmented_best['val_dice']:.4f}")

improvement = (
    augmented_best["val_dice"] -
    baseline_best["val_dice"]
)

print(f"\nDice Improvement: {improvement:.4f}")

In [ ]:

plt.figure(figsize=(10, 6))

plt.plot(
    baseline_df["epoch"],
    baseline_df["val_dice"],
    label="Baseline U-Net",
    linewidth=2
)

plt.plot(
    augmented_df["epoch"],
    augmented_df["val_dice"],
    label="Augmented U-Net",
    linewidth=2
)

# Highlight best results
plt.scatter(
    baseline_best["epoch"],
    baseline_best["val_dice"],
    marker="o",
    s=80,
    zorder=5
)

plt.scatter(
    augmented_best["epoch"],
    augmented_best["val_dice"],
    marker="o",
    s=80,
    zorder=5
)

plt.xlabel("Epoch")
plt.ylabel("Validation Dice Score")
plt.title("Baseline vs Augmented U-Net: Validation Dice")

plt.ylim(0, 1)
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()

plt.savefig(
    PLOTS_DIR / "comparison_val_dice.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:

plt.figure(figsize=(10, 6))

plt.plot(
    baseline_df["epoch"],
    baseline_df["val_loss"],
    label="Baseline U-Net",
    linewidth=2
)

plt.plot(
    augmented_df["epoch"],
    augmented_df["val_loss"],
    label="Augmented U-Net",
    linewidth=2
)

plt.xlabel("Epoch")
plt.ylabel("Validation Loss")
plt.title("Baseline vs Augmented U-Net: Validation Loss")

plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()

plt.savefig(
    PLOTS_DIR / "comparison_val_loss.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:

import sys
import numpy as np
import pandas as pd
import torch

from pathlib import Path
from torch.utils.data import DataLoader

# Find project root
current_dir = Path.cwd().resolve()

ROOT_DIR = next(
    (p for p in [current_dir, *current_dir.parents]
     if (p / "config.py").exists()),
    None
)

if ROOT_DIR is None:
    raise FileNotFoundError("Project root not found!")

if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

from config import SPLITS_FILE, CHECKPOINT_DIR, BATCH_SIZE
from src.dataset import PolypDataset
from src.model import UNetSmall

# Prepare validation data
split_df = pd.read_csv(SPLITS_FILE)
val_df = split_df[split_df["split"] == "val"]

val_dataset = PolypDataset(val_df)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0
)

device = torch.device("cpu")

# Thresholds to evaluate
thresholds = np.round(
    np.arange(0.10, 0.91, 0.05), 2
)

# Model checkpoints
checkpoints = {
    "Baseline V2": CHECKPOINT_DIR / "baseline_v2_best.pth",
    "Augmented U-Net": CHECKPOINT_DIR / "augmented_best.pth"
}

results = {}

# Evaluate each model
for model_name, checkpoint_path in checkpoints.items():

    model = UNetSmall().to(device)

    model.load_state_dict(
        torch.load(
            checkpoint_path,
            map_location=device,
            weights_only=True
        )
    )

    model.eval()

    dice_totals = np.zeros(len(thresholds))
    total_samples = 0

    with torch.inference_mode():

        for images, masks in val_loader:

            images = images.to(device)
            masks = masks.to(device)

            probs = torch.sigmoid(model(images))

            for i, threshold in enumerate(thresholds):

                preds = (probs > threshold).float()

                intersection = (
                    preds * masks
                ).sum(dim=(1, 2, 3))

                denominator = (
                    preds.sum(dim=(1, 2, 3)) +
                    masks.sum(dim=(1, 2, 3))
                )

                dice = (
                    (2 * intersection + 1e-6) /
                    (denominator + 1e-6)
                )

                dice_totals[i] += dice.sum().item()

            total_samples += images.size(0)

    results[model_name] = dice_totals / total_samples

# Display results
results_df = pd.DataFrame({
    "Threshold": thresholds,
    "Baseline Dice": results["Baseline V2"],
    "Augmented Dice": results["Augmented U-Net"]
})

print(results_df.to_string(index=False, float_format="%.4f"))

# Find best threshold for each model
for model_name, scores in results.items():

    best_idx = np.argmax(scores)

    print(f"\n{model_name}")
    print(f"Best Threshold: {thresholds[best_idx]:.2f}")
    print(f"Best Validation Dice: {scores[best_idx]:.4f}")

In [ ]:

import json

model_selection = {
    "selection_dataset": "validation",
    "selected_model": "augmented",
    "models": {
        "baseline_v2": {
            "checkpoint": "outputs/checkpoints/baseline_v2_best.pth",
            "threshold": 0.55,
            "validation_dice": 0.7240
        },
        "augmented": {
            "checkpoint": "outputs/checkpoints/augmented_best.pth",
            "threshold": 0.30,
            "validation_dice": 0.7592
        }
    }
}

output_path = (
    ROOT_DIR / "outputs" / "metrics" / "model_selection.json"
)

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(model_selection, f, indent=4)

print("Model selection saved successfully!")
print("Selected model:", model_selection["selected_model"])
print("File:", output_path)

In [ ]:

import sys
import json
import torch
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from pathlib import Path

# Find project root
current_dir = Path.cwd().resolve()

ROOT_DIR = next(
    (
        p for p in [current_dir, *current_dir.parents]
        if (p / "config.py").exists()
    ),
    None
)

if ROOT_DIR is None:
    raise FileNotFoundError("Project root not found!")

if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

from config import SPLITS_FILE, METRICS_DIR, PLOTS_DIR
from src.dataset import PolypDataset
from src.model import UNetSmall

device = torch.device("cpu")

# Load frozen model selection
selection_path = METRICS_DIR / "model_selection.json"

with open(selection_path, "r", encoding="utf-8") as f:
    selection = json.load(f)

# Load Test Dataset
splits_df = pd.read_csv(SPLITS_FILE)

test_df = splits_df[
    splits_df["split"] == "test"
].reset_index(drop=True)

test_dataset = PolypDataset(test_df)

# Load a saved model
def load_model(model_name):

    settings = selection["models"][model_name]

    checkpoint = ROOT_DIR / settings["checkpoint"]

    model = UNetSmall().to(device)

    model.load_state_dict(
        torch.load(
            checkpoint,
            map_location=device,
            weights_only=True
        )
    )

    model.eval()

    return model, float(settings["threshold"])


baseline_model, baseline_threshold = load_model(
    "baseline_v2"
)

augmented_model, augmented_threshold = load_model(
    "augmented"
)

print("Test samples:", len(test_dataset))
print("Baseline threshold:", baseline_threshold)
print("Augmented threshold:", augmented_threshold)
print("Models loaded successfully!")

In [ ]:

# Calculate Dice Score for one image
def calculate_sample_dice(pred, target):

    pred = pred.astype(bool)
    target = target.astype(bool)

    intersection = np.logical_and(pred, target).sum()

    denominator = pred.sum() + target.sum()

    return (2 * intersection + 1e-6) / (
        denominator + 1e-6
    )


# Fixed test sample indices
sample_indices = [0, 50, 99]

fig, axes = plt.subplots(
    len(sample_indices),
    5,
    figsize=(17, 10)
)

with torch.inference_mode():

    for row, idx in enumerate(sample_indices):

        # Load image and mask
        image_tensor, mask_tensor = test_dataset[idx]

        input_tensor = image_tensor.unsqueeze(0).to(device)

        # Baseline prediction
        baseline_probs = torch.sigmoid(
            baseline_model(input_tensor)
        )

        baseline_pred = (
            baseline_probs > baseline_threshold
        )[0, 0].cpu().numpy()

        # Augmented prediction
        augmented_probs = torch.sigmoid(
            augmented_model(input_tensor)
        )

        augmented_pred = (
            augmented_probs > augmented_threshold
        )[0, 0].cpu().numpy()

        # Convert tensors to NumPy
        image = image_tensor.permute(1, 2, 0).numpy()
        true_mask = mask_tensor.squeeze(0).numpy()

        # Calculate per-image Dice
        baseline_dice = calculate_sample_dice(
            baseline_pred,
            true_mask
        )

        augmented_dice = calculate_sample_dice(
            augmented_pred,
            true_mask
        )

        # Original image
        axes[row, 0].imshow(image)
        axes[row, 0].set_title(f"Original | Sample {idx}")

        # Ground Truth
        axes[row, 1].imshow(
            true_mask,
            cmap="gray",
            vmin=0,
            vmax=1
        )
        axes[row, 1].set_title("Ground Truth")

        # Baseline prediction
        axes[row, 2].imshow(
            baseline_pred,
            cmap="gray",
            vmin=0,
            vmax=1
        )
        axes[row, 2].set_title(
            f"Baseline | Dice: {baseline_dice:.3f}"
        )

        # Augmented prediction
        axes[row, 3].imshow(
            augmented_pred,
            cmap="gray",
            vmin=0,
            vmax=1
        )
        axes[row, 3].set_title(
            f"Augmented | Dice: {augmented_dice:.3f}"
        )

        # Augmented overlay
        axes[row, 4].imshow(image)

        axes[row, 4].imshow(
            np.ma.masked_where(
                ~augmented_pred,
                augmented_pred
            ),
            cmap="Reds",
            alpha=0.5,
            vmin=0,
            vmax=1
        )

        axes[row, 4].set_title("Augmented Overlay")

        for ax in axes[row]:
            ax.axis("off")

plt.tight_layout()

# Save comparison figure
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

plt.savefig(
    PLOTS_DIR / "test_prediction_comparison.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print("Prediction visualization saved!")

In [ ]:

import numpy as np
import pandas as pd
import torch

# Calculate Dice for a single image
def sample_dice(pred, target):
    pred = pred.astype(bool)
    target = target.astype(bool)

    intersection = np.logical_and(pred, target).sum()
    denominator = pred.sum() + target.sum()

    if denominator == 0:
        return 1.0

    return 2 * intersection / denominator


records = []

baseline_model.eval()
augmented_model.eval()

with torch.inference_mode():

    for idx in range(len(test_dataset)):

        image, mask = test_dataset[idx]
        x = image.unsqueeze(0).to(device)

        # Predictions using frozen thresholds
        baseline_prob = torch.sigmoid(
            baseline_model(x)
        )

        augmented_prob = torch.sigmoid(
            augmented_model(x)
        )

        baseline_pred = (
            baseline_prob > baseline_threshold
        )[0, 0].cpu().numpy()

        augmented_pred = (
            augmented_prob > augmented_threshold
        )[0, 0].cpu().numpy()

        true_mask = mask[0].numpy()

        # Calculate per-image metrics
        baseline_dice = sample_dice(
            baseline_pred, true_mask
        )

        augmented_dice = sample_dice(
            augmented_pred, true_mask
        )

        records.append({
            "image_id": test_df.iloc[idx]["image_id"],
            "ground_truth_area_pct": true_mask.mean() * 100,
            "baseline_dice": baseline_dice,
            "augmented_dice": augmented_dice,
            "baseline_pred_area_pct": baseline_pred.mean() * 100,
            "augmented_pred_area_pct": augmented_pred.mean() * 100
        })


# Create DataFrame
error_df = pd.DataFrame(records)

error_df["dice_improvement"] = (
    error_df["augmented_dice"] -
    error_df["baseline_dice"]
)

# Count improvements
better = (error_df["dice_improvement"] > 1e-6).sum()
worse = (error_df["dice_improvement"] < -1e-6).sum()
equal = len(error_df) - better - worse

print("Total Test Images:", len(error_df))
print("Augmented Better:", better)
print("Baseline Better:", worse)
print("Approximately Equal:", equal)

print("\nMean Dice Improvement:")
print(f"{error_df['dice_improvement'].mean():.4f}")

# Identify difficult examples
print("\n5 Worst Augmented Predictions:")

print(
    error_df.nsmallest(5, "augmented_dice")[
        [
            "image_id",
            "ground_truth_area_pct",
            "baseline_dice",
            "augmented_dice"
        ]
    ].to_string(index=False)
)

# Analyze polyp size groups
error_df["polyp_size"] = pd.cut(
    error_df["ground_truth_area_pct"],
    bins=[0, 5, 20, 100],
    labels=["Small", "Medium", "Large"],
    include_lowest=True
)

print("\nPerformance by Polyp Size:")

print(
    error_df.groupby(
        "polyp_size", observed=True
    ).agg(
        count=("image_id", "count"),
        baseline_dice=("baseline_dice", "mean"),
        augmented_dice=("augmented_dice", "mean")
    ).round(4)
)

# Save analysis
output_path = METRICS_DIR / "test_error_analysis.csv"

error_df.to_csv(output_path, index=False)

print("\nError analysis saved successfully!")